# 🎮 Player Churn Analysis and Prediction

In this notebook, we explore the `online_gaming_behavior_dataset.csv`. 
Our goal is to understand what factors lead to a player being 'highly engaged' versus 'churning' (low engagement), and to build a predictive model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('online_gaming_behavior_dataset.csv')
df.head()

### Data Cleaning and Preprocessing
We will drop the `PlayerID` and map the target variable `EngagementLevel` to a binary Churn indicator. Low engagement means the player is churning.

In [ ]:
df = df.drop(columns=['PlayerID'])
df['Churn'] = df['EngagementLevel'].apply(lambda x: 1 if x == 'Low' else 0)
df = df.drop(columns=['EngagementLevel'])

### Exploratory Data Analysis (EDA)
Let's visualize some key factors that influence player churn.

In [ ]:
plt.figure(figsize=(15, 5))

# Churn rate by Genre
plt.subplot(1, 3, 1)
sns.barplot(data=df, x='GameGenre', y='Churn')
plt.title('Churn Rate by Game Genre')
plt.xticks(rotation=45)

# Churn rate by Game Difficulty
plt.subplot(1, 3, 2)
sns.barplot(data=df, x='GameDifficulty', y='Churn')
plt.title('Churn Rate by Game Difficulty')

# Churn rate by In-Game Purchases
plt.subplot(1, 3, 3)
sns.barplot(data=df, x='InGamePurchases', y='Churn')
plt.title('Churn Rate by In-Game Purchases')

plt.tight_layout()
plt.show()

# Additional churn summaries
churn_rate = df['Churn'].mean()
print(f"Overall churn rate: {churn_rate:.2%}")

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

sns.countplot(data=df, x='Churn', ax=axes[0,0])
axes[0,0].set_title('Churn Count')

sns.barplot(data=df, x='Gender', y='Churn', ax=axes[0,1])
axes[0,1].set_title('Churn Rate by Gender')

sns.barplot(data=df, x='Location', y='Churn', ax=axes[1,0])
axes[1,0].set_title('Churn Rate by Location')
axes[1,0].tick_params(axis='x', rotation=45)

sns.histplot(data=df, x='PlayTimeHours', hue='Churn', element='step', stat='density', common_norm=False, ax=axes[1,1])
axes[1,1].set_title('Play Time Distribution by Churn')

plt.tight_layout()
plt.show()

### Additional Insights: Deep Dive Visualizations
Comprehensive analysis of churn patterns across different player characteristics.

In [ ]:
# Multi-dimensional churn analysis
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Churn by Game Difficulty
sns.countplot(data=df, x='GameDifficulty', hue='Churn', ax=axes[0,0])
axes[0,0].set_title('Player Distribution by Game Difficulty & Churn', fontsize=12, fontweight='bold')
axes[0,0].set_ylabel('Count')

# 2. Churn by In-Game Purchases
sns.countplot(data=df, x='InGamePurchases', hue='Churn', ax=axes[0,1])
axes[0,1].set_title('Player Distribution by Purchase Behavior & Churn', fontsize=12, fontweight='bold')
axes[0,1].set_ylabel('Count')
axes[0,1].set_xlabel('In-Game Purchases')

# 3. Churn by Sessions Per Week
sessions_bins = pd.cut(df['SessionsPerWeek'], bins=5)
sns.countplot(data=df, x=sessions_bins, hue='Churn', ax=axes[0,2])
axes[0,2].set_title('Player Distribution by Weekly Sessions & Churn', fontsize=12, fontweight='bold')
axes[0,2].set_ylabel('Count')
axes[0,2].set_xlabel('Sessions Per Week (Binned)')
axes[0,2].tick_params(axis='x', rotation=45)

# 4. Average Play Time by Churn Status
sns.barplot(data=df, x='Churn', y='PlayTimeHours', ax=axes[1,0], palette=['green', 'red'])
axes[1,0].set_title('Average Play Time by Churn Status', fontsize=12, fontweight='bold')
axes[1,0].set_ylabel('Play Time (Hours)')
axes[1,0].set_xticklabels(['Engaged', 'Churned'])

# 5. Average Achievements by Churn Status
sns.barplot(data=df, x='Churn', y='AchievementsUnlocked', ax=axes[1,1], palette=['green', 'red'])
axes[1,1].set_title('Average Achievements by Churn Status', fontsize=12, fontweight='bold')
axes[1,1].set_ylabel('Achievements Unlocked')
axes[1,1].set_xticklabels(['Engaged', 'Churned'])

# 6. Average Session Duration by Churn Status
sns.barplot(data=df, x='Churn', y='AvgSessionDurationMinutes', ax=axes[1,2], palette=['green', 'red'])
axes[1,2].set_title('Average Session Duration by Churn Status', fontsize=12, fontweight='bold')
axes[1,2].set_ylabel('Session Duration (Minutes)')
axes[1,2].set_xticklabels(['Engaged', 'Churned'])

plt.tight_layout()
plt.show()

print("Churn Statistics by Key Metrics:")
print(f"Players with purchases - Churn rate: {df[df['InGamePurchases']==1]['Churn'].mean():.2%}")
print(f"Players without purchases - Churn rate: {df[df['InGamePurchases']==0]['Churn'].mean():.2%}")

### Churn Rate Analysis by Game Characteristics
Understanding which game types and difficulty levels affect player retention.

In [ ]:
# Detailed churn rate analysis by categories
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Churn rate by Genre
genre_churn = df.groupby('GameGenre')['Churn'].agg(['sum', 'count', 'mean']).reset_index()
genre_churn.columns = ['Genre', 'Churned', 'Total', 'ChurnRate']
sns.barplot(data=genre_churn, x='Genre', y='ChurnRate', ax=axes[0,0], palette='coolwarm')
axes[0,0].set_title('Churn Rate by Game Genre', fontsize=12, fontweight='bold')
axes[0,0].set_ylabel('Churn Rate')
axes[0,0].set_ylim([0, 1])
for i, v in enumerate(genre_churn['ChurnRate']):
    axes[0,0].text(i, v + 0.02, f'{v:.1%}', ha='center')

# 2. Churn rate by Difficulty
diff_churn = df.groupby('GameDifficulty')['Churn'].agg(['sum', 'count', 'mean']).reset_index()
diff_churn.columns = ['Difficulty', 'Churned', 'Total', 'ChurnRate']
sns.barplot(data=diff_churn, x='Difficulty', y='ChurnRate', ax=axes[0,1], palette='RdYlGn_r')
axes[0,1].set_title('Churn Rate by Game Difficulty', fontsize=12, fontweight='bold')
axes[0,1].set_ylabel('Churn Rate')
axes[0,1].set_ylim([0, 1])
for i, v in enumerate(diff_churn['ChurnRate']):
    axes[0,1].text(i, v + 0.02, f'{v:.1%}', ha='center')

# 3. Churn rate by Gender
gender_churn = df.groupby('Gender')['Churn'].agg(['sum', 'count', 'mean']).reset_index()
gender_churn.columns = ['Gender', 'Churned', 'Total', 'ChurnRate']
sns.barplot(data=gender_churn, x='Gender', y='ChurnRate', ax=axes[1,0], palette='Set2')
axes[1,0].set_title('Churn Rate by Gender', fontsize=12, fontweight='bold')
axes[1,0].set_ylabel('Churn Rate')
axes[1,0].set_ylim([0, 1])
for i, v in enumerate(gender_churn['ChurnRate']):
    axes[1,0].text(i, v + 0.02, f'{v:.1%}', ha='center')

# 4. Churn rate by Location
location_churn = df.groupby('Location')['Churn'].agg(['sum', 'count', 'mean']).reset_index()
location_churn.columns = ['Location', 'Churned', 'Total', 'ChurnRate']
sns.barplot(data=location_churn, x='Location', y='ChurnRate', ax=axes[1,1], palette='viridis')
axes[1,1].set_title('Churn Rate by Location', fontsize=12, fontweight='bold')
axes[1,1].set_ylabel('Churn Rate')
axes[1,1].set_ylim([0, 1])
for i, v in enumerate(location_churn['ChurnRate']):
    axes[1,1].text(i, v + 0.02, f'{v:.1%}', ha='center')

plt.tight_layout()
plt.show()

### Age & Experience Analysis
How player age and progression affect churn patterns.

In [ ]:
# Age groups and experience analysis
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Age distribution by churn status
df['AgeGroup'] = pd.cut(df['Age'], bins=[0, 18, 30, 45, 60], labels=['<18', '18-30', '30-45', '45+'])
age_churn = df.groupby('AgeGroup')['Churn'].agg(['sum', 'count', 'mean']).reset_index()
age_churn.columns = ['AgeGroup', 'Churned', 'Total', 'ChurnRate']
sns.barplot(data=age_churn, x='AgeGroup', y='ChurnRate', ax=axes[0,0], palette='husl')
axes[0,0].set_title('Churn Rate by Age Group', fontsize=12, fontweight='bold')
axes[0,0].set_ylabel('Churn Rate')
axes[0,0].set_ylim([0, 1])
for i, v in enumerate(age_churn['ChurnRate']):
    axes[0,0].text(i, v + 0.02, f'{v:.1%}', ha='center')

# 2. Player Level distribution by churn
level_churn = df.groupby(pd.cut(df['PlayerLevel'], bins=5))['Churn'].mean()
axes[0,1].bar(range(len(level_churn)), level_churn.values, color='steelblue', edgecolor='black')
axes[0,1].set_title('Churn Rate by Player Level (Binned)', fontsize=12, fontweight='bold')
axes[0,1].set_ylabel('Churn Rate')
axes[0,1].set_xlabel('Player Level Bins')
axes[0,1].set_ylim([0, 1])

# 3. Achievements vs Churn
achievement_bins = pd.cut(df['AchievementsUnlocked'], bins=5)
achievement_churn = df.groupby(achievement_bins)['Churn'].agg(['sum', 'count', 'mean']).reset_index()
achievement_churn.columns = ['Achievements', 'Churned', 'Total', 'ChurnRate']
axes[1,0].bar(range(len(achievement_churn)), achievement_churn['ChurnRate'].values, color='coral', edgecolor='black')
axes[1,0].set_title('Churn Rate by Achievements Unlocked (Binned)', fontsize=12, fontweight='bold')
axes[1,0].set_ylabel('Churn Rate')
axes[1,0].set_xlabel('Achievement Bins')
axes[1,0].set_ylim([0, 1])

# 4. Sessions per week impact on churn
session_bins = pd.cut(df['SessionsPerWeek'], bins=5)
session_churn = df.groupby(session_bins)['Churn'].agg(['sum', 'count', 'mean']).reset_index()
session_churn.columns = ['Sessions', 'Churned', 'Total', 'ChurnRate']
axes[1,1].bar(range(len(session_churn)), session_churn['ChurnRate'].values, color='mediumseagreen', edgecolor='black')
axes[1,1].set_title('Churn Rate by Sessions Per Week (Binned)', fontsize=12, fontweight='bold')
axes[1,1].set_ylabel('Churn Rate')
axes[1,1].set_xlabel('Session Frequency Bins')
axes[1,1].set_ylim([0, 1])

plt.tight_layout()
plt.show()

### Engagement Metrics Summary
Key statistics showing player engagement patterns.

In [ ]:
# Comprehensive engagement metrics
print("\n" + "="*80)
print("PLAYER CHURN ANALYSIS SUMMARY")
print("="*80 + "\n")

# Overall statistics
print("OVERALL STATISTICS:")
print(f"Total Players: {len(df)}")
print(f"Churned Players: {df['Churn'].sum()} ({df['Churn'].mean():.2%})")
print(f"Engaged Players: {(1-df['Churn']).sum()} ({(1-df['Churn']).mean():.2%})\n")

# Comparison of engaged vs churned
print("COMPARISON: ENGAGED vs CHURNED PLAYERS")
print("-" * 80)
metrics = ['Age', 'PlayTimeHours', 'SessionsPerWeek', 'AvgSessionDurationMinutes', 'PlayerLevel', 'AchievementsUnlocked']
comparison_df = df.groupby('Churn')[metrics].mean()
comparison_df.index = ['Engaged', 'Churned']
print(comparison_df.round(2))

# Display as visual table
fig, ax = plt.subplots(figsize=(14, 4))
ax.axis('tight')
ax.axis('off')

table_data = []
table_data.append(['Metric'] + list(comparison_df.index))
for metric in metrics:
    row = [metric]
    for churn_status in comparison_df.index:
        row.append(f"{comparison_df.loc[churn_status, metric]:.2f}")
    table_data.append(row)

table = ax.table(cellText=table_data, cellLoc='center', loc='center', 
                colWidths=[0.2, 0.3, 0.3])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2)

# Style header
for i in range(len(table_data[0])):
    table[(0, i)].set_facecolor('#40466e')
    table[(0, i)].set_text_props(weight='bold', color='white')

# Alternate row colors
for i in range(1, len(table_data)):
    for j in range(len(table_data[0])):
        if i % 2 == 0:
            table[(i, j)].set_facecolor('#f0f0f0')
        else:
            table[(i, j)].set_facecolor('white')

plt.title('Key Metrics: Engaged vs Churned Players', fontsize=14, fontweight='bold', pad=20)
plt.show()

print("\n" + "="*80)
print("KEY INSIGHTS:")
print("="*80)
print(f"✓ Churned players have {comparison_df.loc['Engaged', 'PlayTimeHours'] - comparison_df.loc['Churned', 'PlayTimeHours']:.1f} fewer hours of play time")
print(f"✓ Churned players play {comparison_df.loc['Engaged', 'SessionsPerWeek'] - comparison_df.loc['Churned', 'SessionsPerWeek']:.1f} fewer sessions per week")
print(f"✓ Churned players have {comparison_df.loc['Engaged', 'AchievementsUnlocked'] - comparison_df.loc['Churned', 'AchievementsUnlocked']:.1f} fewer achievements")
print(f"✓ Churned players have {comparison_df.loc['Engaged', 'PlayerLevel'] - comparison_df.loc['Churned', 'PlayerLevel']:.1f} lower player level")
print("="*80 + "\n")

Players with higher difficulty or lack of in-game purchases might have different churn patterns. Next, let's explore continuous variables like Play Time and Age.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(data=df, x='Churn', y='PlayTimeHours', ax=axes[0])
axes[0].set_title('Play Time vs Churn')

sns.boxplot(data=df, x='Churn', y='Age', ax=axes[1])
axes[1].set_title('Age vs Churn')

plt.tight_layout()
plt.show()


Finally, let's look at the correlation among continuous variables and our target.

In [ ]:
numerical_features = ['Age', 'PlayTimeHours', 'SessionsPerWeek', 'AvgSessionDurationMinutes', 'PlayerLevel', 'AchievementsUnlocked', 'Churn']
corr = df[numerical_features].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

# pairwise scatter/hist relationships
sns.pairplot(df[numerical_features], hue='Churn', corner=True, diag_kind='hist')
plt.suptitle('Pairwise Relationships Among Numerical Features', y=1.02)
plt.show()

### Encoding Categorical Variables
We convert categorical features like Gender, Location, and Genre to numeric.

In [3]:
gender_map = {val: idx for idx, val in enumerate(df['Gender'].unique())}
df['Gender'] = df['Gender'].map(gender_map)

location_map = {val: idx for idx, val in enumerate(df['Location'].unique())}
df['Location'] = df['Location'].map(location_map)

genre_map = {val: idx for idx, val in enumerate(df['GameGenre'].unique())}
df['GameGenre'] = df['GameGenre'].map(genre_map)

diff_map = {val: idx for idx, val in enumerate(df['GameDifficulty'].unique())}
df['GameDifficulty'] = df['GameDifficulty'].map(diff_map)

### Model Training
Let's split the dataset, scale our features, and train a Random Forest Classifier.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

### Feature Importance
What features drive player churn the most?

In [5]:
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]
plt.figure(figsize=(10, 6))
sns.barplot(x=importances[indices], y=X.columns[indices])
plt.title('Feature Importances for Player Churn')
plt.show()

### Save Model and Scaler

In [6]:
import joblib
joblib.dump(model, 'model.pkl')
joblib.dump(scaler, 'scaler.pkl')
print('Models saved successfully.')